# Descarga de datos desde datos.gov.co

Este notebook descarga el dataset `p6dx-8zbt` usando la API de Socrata, carga el resultado en un `DataFrame`, muestra hasta 1000 filas y lo guarda en `Data/Raw/secop_procesos.parquet` dentro del proyecto.

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from requests.exceptions import HTTPError
from sodapy import Socrata

In [2]:
def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / ".env").exists() and (candidate / "Data").exists() and (candidate / "Code").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
load_dotenv(PROJECT_ROOT / ".env")

DOMAIN = os.getenv("SOCRATA_DOMAIN", "www.datos.gov.co")
DATASET_ID = os.getenv("SOCRATA_DATASET_ID", "p6dx-8zbt")
APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN") or None
API_SECRET = os.getenv("SOCRATA_API_SECRET")

OUTPUT_PATH = PROJECT_ROOT / "Data" / "Raw" / "secop_procesos.parquet"
PAGE_SIZE = 1000
TARGET_PREVIEW_ROWS = 1000
MAX_ROWS_TO_DOWNLOAD = 1000  # Usa None si despues quieres bajar el dataset completo.

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Dominio: {DOMAIN}")
print(f"Dataset: {DATASET_ID}")
print(f"Archivo de salida: {OUTPUT_PATH.resolve()}")
print(f"App token cargado: {'sí' if APP_TOKEN else 'no'}")
print(f"Secret cargada: {'sí' if API_SECRET else 'no'}")

Project root: C:\Users\nesto\OneDrive - Universidad de la Sabana\Universidad\2026-1\Big data\Proyecto
Dominio: www.datos.gov.co
Dataset: p6dx-8zbt
Archivo de salida: C:\Users\nesto\OneDrive - Universidad de la Sabana\Universidad\2026-1\Big data\Proyecto\Data\Raw\secop_procesos.parquet
App token cargado: sí
Secret cargada: sí


In [3]:
def download_dataset(domain: str, dataset_id: str, app_token: str | None, page_size: int = 1000, max_rows: int | None = 1000) -> pd.DataFrame:
    """Descarga el dataset completo por páginas y lo devuelve como DataFrame."""
    def _fetch_rows(token: str | None) -> list[dict]:
        rows = []
        offset = 0

        with Socrata(domain, token, timeout=60) as client:
            while True:
                current_limit = page_size
                if max_rows is not None:
                    remaining = max_rows - len(rows)
                    if remaining <= 0:
                        break
                    current_limit = min(page_size, remaining)

                batch = client.get(dataset_id, limit=current_limit, offset=offset)

                if not batch:
                    break

                rows.extend(batch)
                print(f"Descargados {len(rows)} registros...")

                if len(batch) < current_limit:
                    break

                offset += current_limit

        return rows

    try:
        all_rows = _fetch_rows(app_token)
    except HTTPError as exc:
        if exc.response is not None and exc.response.status_code == 403 and app_token:
            print("El app token fue rechazado. Reintentando sin token porque el dataset es publico...")
            all_rows = _fetch_rows(None)
        else:
            raise

    return pd.DataFrame.from_records(all_rows)

In [4]:
df = download_dataset(DOMAIN, DATASET_ID, APP_TOKEN, page_size=PAGE_SIZE, max_rows=MAX_ROWS_TO_DOWNLOAD)

print(f"Total de filas descargadas: {len(df):,}")
print(f"Total de columnas: {len(df.columns)}")

df.to_parquet(OUTPUT_PATH, index=False)
print(f"Parquet guardado en: {OUTPUT_PATH.resolve()}")

preview_df = df.head(TARGET_PREVIEW_ROWS)
preview_df

El app token fue rechazado. Reintentando sin token porque el dataset es publico...
Descargados 1000 registros...
Total de filas descargadas: 1,000
Total de columnas: 57
Parquet guardado en: C:\Users\nesto\OneDrive - Universidad de la Sabana\Universidad\2026-1\Big data\Proyecto\Data\Raw\secop_procesos.parquet


,entidad,nit_entidad,departamento_entidad,ciudad_entidad,ordenentidad,codigo_pci,id_del_proceso,referencia_del_proceso,ppi,id_del_portafolio,...,categorias_adicionales,urlproceso,codigo_entidad,estado_resumen,fecha_de_recepcion_de,fecha_de_apertura_de_respuesta,fecha_de_apertura_efectiva,fecha_adjudicacion,fecha_de_publicacion_fase_2,fecha_de_publicacion
0,DEPARTAMENTO ADMINISTRATIVO NACIONAL DE ESTADI...,899999027,Distrito Capital de Bogotá,Bogotá,Nacional,Centralizada,CO1.REQ.2577563,EDP-545-2022,700474109,CO1.BDOS.2503732,...,No definido,{'url': 'https://community.secop.gov.co/Public...,700474109,Presentación de oferta,NaN,NaN,NaN,NaN,NaN,NaN
1,ALCALDIA LOCAL DE SUMAPAZ,899999061,Distrito Capital de Bogotá,Bogotá,Territorial,Centralizada,CO1.REQ.5912737,FDRSCD-064-2024 (103702),702096124,CO1.BDOS.5796124,...,No definido,{'url': 'https://community.secop.gov.co/Public...,702096124,Presentación de oferta,NaN,NaN,NaN,NaN,NaN,NaN
2,CENAC AVIACION,830039207,Distrito Capital de Bogotá,No Definido,Nacional,Centralizada,CO1.REQ.8610154,SASD-191-CENACAVIACION-2025 (Manifestación de ...,702359944,CO1.BDOS.8355416,...,No definido,{'url': 'https://community.secop.gov.co/Public...,702359944,Adjudicado,2025-07-25T00:00:00.000,2025-07-25T00:00:00.000,2025-07-25T00:00:00.000,2025-08-19T00:00:00.000,NaN,NaN
3,ISVIMED,900014480,Antioquia,Medellín,Territorial,Centralizada,CO1.REQ.9756253,175-2026,703835165,CO1.BDOS.9589062,...,No definido,{'url': 'https://community.secop.gov.co/Public...,703835165,Presentación de oferta,NaN,NaN,NaN,NaN,NaN,NaN
4,COLEGIO GABRIEL BETANCOURT MEJIA I.E.D,900143276,Distrito Capital de Bogotá,No Definido,Territorial,Centralizada,CO1.REQ.7128715,RE-CGBM-049-2024,703916536,CO1.BDOS.6995772,...,No definido,{'url': 'https://community.secop.gov.co/Public...,703916536,Presentación de oferta,2024-11-12T00:00:00.000,2024-11-13T00:00:00.000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,IGAC Sede Central *,899999004,Distrito Capital de Bogotá,No Definido,Nacional,Centralizada,CO1.REQ.9707173,CD-681-2026-CAQ,700663107,CO1.BDOS.9539955,...,No definido,{'url': 'https://community.secop.gov.co/Public...,700663107,Presentación de oferta,NaN,NaN,NaN,NaN,NaN,NaN
996,DEPARTAMENTO DE POLICIA QUINDIO,800140986,Quindío,No Definido,Territorial,Centralizada,CO1.REQ.1728739,DEQUI-GRULO 002,701703761,CO1.BDOS.1678570,...,V115101505,{'url': 'https://community.secop.gov.co/Public...,701703761,Presentación de oferta,2021-01-27T00:00:00.000,NaN,2021-01-20T00:00:00.000,NaN,NaN,NaN
997,CENTRO DE SALUD HERMES ANDRADE MEJIA E.S.E TAN...,900125582,Nariño,Tangua,Territorial,Centralizada,CO1.REQ.7630754,2025000051,701950016,CO1.BDOS.7495455,...,No definido,{'url': 'https://community.secop.gov.co/Public...,701950016,Presentación de oferta,NaN,NaN,NaN,NaN,NaN,NaN
998,MUNICIPIO DE CHIGORODO,890980998,Antioquia,Chigorodó,Territorial,Centralizada,CO1.REQ.3080983,199-2022 PRESTACION DE SERVICIOS,704891027,CO1.BDOS.2998618,...,No definido,{'url': 'https://community.secop.gov.co/Public...,704891027,Fase de ofertas,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Si quieres revisar únicamente las primeras columnas:
preview_df.iloc[:, :10].head()

,entidad,nit_entidad,departamento_entidad,ciudad_entidad,ordenentidad,codigo_pci,id_del_proceso,referencia_del_proceso,ppi,id_del_portafolio
0,DEPARTAMENTO ADMINISTRATIVO NACIONAL DE ESTADI...,899999027,Distrito Capital de Bogotá,Bogotá,Nacional,Centralizada,CO1.REQ.2577563,EDP-545-2022,700474109,CO1.BDOS.2503732
1,ALCALDIA LOCAL DE SUMAPAZ,899999061,Distrito Capital de Bogotá,Bogotá,Territorial,Centralizada,CO1.REQ.5912737,FDRSCD-064-2024 (103702),702096124,CO1.BDOS.5796124
2,CENAC AVIACION,830039207,Distrito Capital de Bogotá,No Definido,Nacional,Centralizada,CO1.REQ.8610154,SASD-191-CENACAVIACION-2025 (Manifestación de ...,702359944,CO1.BDOS.8355416
3,ISVIMED,900014480,Antioquia,Medellín,Territorial,Centralizada,CO1.REQ.9756253,175-2026,703835165,CO1.BDOS.9589062
4,COLEGIO GABRIEL BETANCOURT MEJIA I.E.D,900143276,Distrito Capital de Bogotá,No Definido,Territorial,Centralizada,CO1.REQ.7128715,RE-CGBM-049-2024,703916536,CO1.BDOS.6995772
